### 1. Встановіть `azure-ai-inference`

In [1]:
%pip install azure-ai-inference

Note: you may need to restart the kernel to use updated packages.


### 2. Імпортуйте допоміжні бібліотеки та створіть облікові дані

In [2]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv

load_dotenv()
token = os.getenv("GITHUB_TOKEN")
endpoint = "https://models.inference.ai.azure.com"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)

### 3. Пошук правильної моделі  
Моделі GPT-3.5-turbo або GPT-4 можуть розуміти та генерувати природну мову.

In [3]:
# Виберіть модель загального призначення curie для тексту
model_name = "gpt-4o"

## 4. Дизайн промпту  

"Магія великих мовних моделей полягає в тому, що, навчаючись мінімізувати цю похибку передбачення на великих обсягах тексту, моделі в результаті вивчають концепції, корисні для цих передбачень. Наприклад, вони вивчають такі концепції як"(1):

* як правильно писати
* як працює граматика
* як перефразовувати
* як відповідати на запитання
* як вести розмову
* як писати багатьма мовами
* як кодувати
* тощо.

#### Як керувати великою мовною моделлю  
"З усіх входів до великої мовної моделі, безумовно, найбільш впливовим є текстовий промпт"(1).

Великі мовні моделі можна спонукати до створення виводу кількома способами:

Інструкція: Скажіть моделі, що ви хочете
Завершення: Спонукайте модель завершити початок того, що ви хочете
Демонстрація: Покажіть моделі, що ви хочете, за допомогою:
Кількох прикладів у промпті
Багатьох сотень або тисяч прикладів у навчальному наборі даних для тонкого налаштування"

#### Існують три основні рекомендації щодо створення промптів:

**Показуйте та розповідайте**. Чітко вказуйте, що ви хочете, через інструкції, приклади або їх комбінацію. Якщо ви хочете, щоб модель розташувала список елементів в алфавітному порядку або класифікувала абзац за настроєм, покажіть їй, що саме ви хочете.

**Надавайте якісні дані**. Якщо ви намагаєтеся створити класифікатор або змусити модель слідувати певній схемі, переконайтеся, що є достатньо прикладів. Обов'язково перевірте свої приклади — модель зазвичай достатньо розумна, щоб зрозуміти основні орфографічні помилки та дати вам відповідь, але вона також може припустити, що це навмисно, і це може вплинути на відповідь.

**Перевіряйте налаштування**. Параметри temperature та top_p контролюють, наскільки детермінованою є модель при генерації відповіді. Якщо ви просите відповідь, де є лише одна правильна відповідь, то вам варто встановити їх нижче. Якщо ви шукаєте більш різноманітні відповіді, то можливо вам захочеться встановити їх вище. Найпоширенішою помилкою, яку люди роблять з цими налаштуваннями, є припущення, що вони є елементами керування "розумністю" або "креативністю".

### 5. Надсилаємо!

In [4]:
# Створіть свій перший промпт
text_prompt = "Should oxford commas always be used?"

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

'The use of the Oxford comma (also known as the serial comma) is a matter of style and preference, and whether it should always be used depends on the context and the style guide being followed. The Oxford comma is the comma placed before the final conjunction (such as "and" or "or") in a list of three or more items. For example:\n\n- With Oxford comma: "I ate apples, oranges, and bananas."\n- Without Oxford comma: "I ate apples, oranges and bananas."\n\nHere are some considerations:\n\n### **Reasons to Use the Oxford Comma**\n1. **Clarity**: The Oxford comma can help avoid ambiguity. For example:\n   - Without Oxford comma: "I’d like to thank my parents, Oprah Winfrey and God." (This could imply that your parents are Oprah Winfrey and God.)\n   - With Oxford comma: "I’d like to thank my parents, Oprah Winfrey, and God." (This makes it clear that your parents are separate from Oprah Winfrey and God.)\n\n2. **Consistency**: Using the Oxford comma consistently makes your writing style un

### Повторіть той самий виклик, як порівнюються результати?

In [5]:
response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

'The use of the Oxford comma (also known as the serial comma) is a matter of style and preference, but in general, it is recommended for clarity in certain situations. Here\'s an overview:\n\n### What is the Oxford Comma?\nThe Oxford comma is the comma that comes before the conjunction (usually "and" or "or") in a list of three or more items. For example:\n- With Oxford comma: "I invited my parents, Taylor Swift, and Harry Styles."\n- Without Oxford comma: "I invited my parents, Taylor Swift and Harry Styles."\n\n### Pros of Using the Oxford Comma\n1. **Clarity**: In some sentences, the lack of an Oxford comma can create ambiguity. For example:\n   - Without Oxford comma: *"I invited two strippers, JFK and Stalin."* (This makes it seem that JFK and Stalin are the two strippers.)\n   - With Oxford comma: *"I invited two strippers, JFK, and Stalin."* (This makes it clear that the strippers, JFK, and Stalin are separate entities.)\n2. **Consistency**: Using the Oxford comma consistently e

## Підсумовування тексту  
#### Завдання  
Підсумуйте текст, додавши 'tl;dr:' в кінці текстового уривку. Зверніть увагу, як модель розуміє, як виконувати низку завдань без додаткових інструкцій. Ви можете експериментувати з більш описовими промптами, ніж tl;dr, щоб модифікувати поведінку моделі та налаштувати підсумок, який ви отримуєте(3).  

Останні роботи продемонстрували значні успіхи в багатьох завданнях та тестах NLP шляхом попереднього навчання на великому корпусі тексту з подальшим тонким налаштуванням на конкретне завдання. Хоча зазвичай архітектура не залежить від завдання, цей метод все ще вимагає наборів даних для тонкого налаштування, специфічних для завдання, які містять тисячі або десятки тисяч прикладів. На відміну від цього, люди зазвичай можуть виконувати нове мовне завдання лише з кількох прикладів або з простих інструкцій - щось, з чим поточні системи NLP все ще значною мірою борються. Тут ми показуємо, що масштабування мовних моделей значно покращує ефективність, не залежну від завдання, з кількома прикладами, іноді навіть досягаючи конкурентоспроможності з попередніми найсучаснішими підходами тонкого налаштування.

Tl;dr

# Вправи для кількох випадків використання  
1. Підсумовування тексту  
2. Класифікація тексту  
3. Генерація нових назв продуктів

In [6]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"

In [7]:
#Встановлення кількох додаткових типових параметрів під час виклику API

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

'Scaling up language models significantly improves their task-agnostic, few-shot performance, making them more competitive with fine-tuned models, and reducing reliance on large task-specific datasets.'

Попереднє навчання великих мовних моделей з подальшим тонким налаштуванням показує чудові результати, але вимагає великих наборів даних для кожного завдання. На відміну від людей, які вчаться з кількох прикладів, сучасні NLP-системи з цим борються. Дослідження демонструє, що простое масштабування мовних моделей значно покращує їхню здатність виконувати нові завдання лише на основі кількох прикладів (few-shot learning), іноді навіть досягаючи рівня спеціально навчених моделей.

## Класифікація тексту  
#### Завдання  
Класифікуйте елементи на категорії, надані під час виведення. У наступному прикладі ми надаємо як категорії, так і текст для класифікації у промпті(*playground_reference). 

Запит клієнта: Привіт, одна з клавіш на клавіатурі мого ноутбука нещодавно зламалася, і мені потрібна заміна:

Класифікована категорія:

In [8]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]

inquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:

Classified category:


In [9]:
#Встановлення кількох додаткових типових параметрів під час виклику API

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

'Hardware Support'

Запит клієнта чітко описує фізичну поломку апаратного компонента (клавіатури ноутбука), що безпосередньо відповідає категорії Hardware Support. Це не стосується питань ціноутворення (Pricing) чи проблем з програмним забезпеченням (Software Support)

## Генерація нових назв продуктів
#### Завдання
Створіть назви продуктів з прикладів слів. Тут ми включаємо в промпт інформацію про продукт, для якого ми збираємося генерувати назви. Ми також надаємо схожий приклад, щоб показати схему, яку ми хочемо отримати. Ми також встановили високе значення температури, щоб збільшити випадковість та отримати більш інноваційні відповіді.

Опис продукту: Домашній міксер для молочних коктейлів
Ключові слова: швидкий, здоровий, компактний.
Назви продуктів: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Опис продукту: Пара взуття, яка може підходити для будь-якого розміру стопи.
Ключові слова: адаптивний, що підходить, omni-fit.

In [10]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

Product description: A home milkshake maker
Seed words: fast, healthy, compact.
Product names: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Product description: A pair of shoes that can fit any foot size.
Seed words: adaptable, fit, omni-fit.


In [ ]:
  #Встановлення кількох додаткових типових параметрів під час виклику API

  response = client.complete(
    model=model_name,
    messages = [{"role":"system", "content":"You are a helpful assistant."},
                {"role":"user","content":prompt}])

  response.choices[0].message.content

'**Product Names: OmniStep, FitAll, FlexiFit, AdaptiShoe**'

Назви:

Використовують ключові слова (adaptable, fit, omni-fit)

Дотримуються тієї ж структури, що й у прикладі (креативні назви через поєднання слів)

Ефективно передають основну функцію продукту

Генерація демонструє здатність моделі розпізнавати патерни та застосовувати їх до нових завдань, що є ключовим для few-shot навчання.

_____

___

### Індивідуальне завдання (Варіант №3)

Розробіть додаток для генерації креативних назв для технологічних продуктів.

In [13]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv

# Завантаження змінних середовища
load_dotenv()
token = os.getenv("GITHUB_TOKEN")
endpoint = "https://models.inference.ai.azure.com"

# Ініціалізація клієнта
client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)

model_name = "gpt-4o"

## Підсумовування тексту  
#### Завдання  
Підсумуйте текст, додавши 'tl;dr:' в кінці текстового уривку. Зверніть увагу, як модель розуміє, як виконувати низку завдань без додаткових інструкцій. Ви можете експериментувати з більш описовими промптами, ніж tl;dr, щоб модифікувати поведінку моделі та налаштувати підсумок, який ви отримуєте(3).  

Останні роботи продемонстрували значні успіхи в багатьох завданнях та тестах NLP шляхом попереднього навчання на великому корпусі тексту з подальшим тонким налаштуванням на конкретне завдання. Хоча зазвичай архітектура не залежить від завдання, цей метод все ще вимагає наборів даних для тонкого налаштування, специфічних для завдання, які містять тисячі або десятки тисяч прикладів. На відміну від цього, люди зазвичай можуть виконувати нове мовне завдання лише з кількох прикладів або з простих інструкцій - щось, з чим поточні системи NLP все ще значною мірою борються. Тут ми показуємо, що масштабування мовних моделей значно покращує ефективність, не залежну від завдання, з кількома прикладами, іноді навіть досягаючи конкурентоспроможності з попередніми найсучаснішими підходами тонкого налаштування.

Tl;dr

# Вправи для кількох випадків використання  
1. Підсумовування тексту  
2. Класифікація тексту  
3. Генерація нових назв продуктів

In [ ]:
text = "ВИМОГИ ДО НАЗВ \n- Короткі та легко вимовні \n- Відображають суть продукту \n- Підходять для глобального ринку \n- Унікальні та запам'ятовувані"

prompt = "Завдання: Згенеруй технологічні продукцію. \n\nНАЗВА: \nОПИС ПРОДУКТУ: \nГАЛУЗЬ: \nКЛЮЧОВІ СЛОВА: \nСТИЛЬ НАЗВ: \n\n" + text + "\n\nПРИКЛАДИ ДЛЯ НАТХНЕННЯ: \n- Для домашнього міксера: HomeShaker, Fit Shaker, QuickShake, Shake Maker \n- Для адаптивного взуття: OmniStep, FitAll, FlexiFit, AdaptiShoe \n- Для AI-асистента: NeuroGuide, SmartMate, IntelAssistant, CogniBot"

print(prompt)

Завдання: Згенеруй технологічні продукцію. 

НАЗВА: 
ОПИС ПРОДУКТУ: 
ГАЛУЗЬ: 
КЛЮЧОВІ СЛОВА: 
СТИЛЬ НАЗВ: 

ВИМОГИ ДО НАЗВ 
- Короткі та легко вимовні 
- Відображають суть продукту 
- Підходять для глобального ринку 
- Унікальні та запам'ятовувані

ПРИКЛАДИ ДЛЯ НАТХНЕННЯ: 
- Для домашнього міксера: HomeShaker, Fit Shaker, QuickShake, Shake Maker 
- Для адаптивного взуття: OmniStep, FitAll, FlexiFit, AdaptiShoe 
- Для AI-асистента: NeuroGuide, SmartMate, IntelAssistant, CogniBot


In [25]:
response = client.complete(
    model=model_name,
        messages=[{"role": "system", "content": "You are a creative branding expert specializing in tech product names."},
                {"role": "user", "content": prompt}])
print(response.choices[0].message.content)

**НАЗВА:** DataPulse  
**ОПИС ПРОДУКТУ:** Інноваційна платформа для аналізу великих обсягів даних у реальному часі, яка допомагає компаніям швидко приймати стратегічні рішення на основі даних.  
**ГАЛУЗЬ:** Big Data, аналітика  
**КЛЮЧОВІ СЛОВА:** дані, аналітика, швидкість, реальний час, рішення  
**СТИЛЬ НАЗВ:** Технологічно, сучасно, динамічно  

---

**НАЗВА:** SkyLink  
**ОПИС ПРОДУКТУ:** Революційна система супутникового інтернету з мінімальною затримкою для віддалених місць та глобального доступу.  
**ГАЛУЗЬ:** Телекомунікації  
**КЛЮЧОВІ СЛОВА:** супутник, швидкість, інтернет, глобальний  
**СТИЛЬ НАЗВ:** Лаконічно, зрозуміло, глобально  

---

**НАЗВА:** BioSense  
**ОПИС ПРОДУКТУ:** Носимий пристрій для швидкого медичного моніторингу (пульс, сатурація, температура, рівень глюкози) із мобільним додатком для аналізу здоров'я.  
**ГАЛУЗЬ:** Медичні технології, носимі пристрої  
**КЛЮЧОВІ СЛОВА:** здоров'я, сенсори, моніторинг, аналіз  
**СТИЛЬ НАЗВ:** Прості, професійні, глобаль

## Класифікація тексту  
#### Завдання  
Класифікуйте елементи на категорії, надані під час виведення. У наступному прикладі ми надаємо як категорії, так і текст для класифікації у промпті(*playground_reference). 

Запит клієнта: Привіт, одна з клавіш на клавіатурі мого ноутбука нещодавно зламалася, і мені потрібна заміна:

Класифікована категорія:

In [30]:
prompt = "Класифікуй наступний запит в одну з категорій: [Big Data, Телекомунікації, Медичні технології, Музична індустрія, AI] \n\nЗапит клієнта: потрібно знайти технологію, що пов'язана з оброкою великою кількості інформації \n\nГалузь:"
print(prompt)

Класифікуй наступний запит в одну з категорій: [Big Data, Телекомунікації, Медичні технології, Музична індустрія, AI] 

Запит клієнта: потрібно знайти технологію, що пов'язана з оброкою великою кількості інформації 

Галузь:


In [31]:
response = client.complete(
    model=model_name,
    messages=[{"role": "system", "content": "Ти допоміжний асистент для класифікації запитів щодо технологічних продуктів."},
        {"role": "user", "content": prompt}])

print(response.choices[0].message.content)

Big Data


## Генерація нових назв продуктів
#### Завдання

In [32]:
prompt = "\nОПИС ПРОДУКТУ: Революційна система супутникового інтернету з мінімальною затримкою для віддалених місць та глобального доступу.  \nГАЛУЗЬ: Телекомунікації  \nКЛЮЧОВІ СЛОВА: супутник, швидкість, інтернет, глобальний\n\n ОПИС ПРОДУКТУ: Носимий пристрій для швидкого медичного моніторингу (пульс, сатурація, температура, рівень глюкози) із мобільним додатком для аналізу здоров'я.  \n ГАЛУЗЬ: Медичні технології, носимі пристрої  \n КЛЮЧОВІ СЛОВА: здоров'я, сенсори, моніторинг, аналіз\n СТИЛЬ НАЗВ: Прості, професійні, глобальні"

print(prompt)


ОПИС ПРОДУКТУ: Революційна система супутникового інтернету з мінімальною затримкою для віддалених місць та глобального доступу.  
ГАЛУЗЬ: Телекомунікації  
КЛЮЧОВІ СЛОВА: супутник, швидкість, інтернет, глобальний

 ОПИС ПРОДУКТУ: Носимий пристрій для швидкого медичного моніторингу (пульс, сатурація, температура, рівень глюкози) із мобільним додатком для аналізу здоров'я.  
 ГАЛУЗЬ: Медичні технології, носимі пристрої  
 КЛЮЧОВІ СЛОВА: здоров'я, сенсори, моніторинг, аналіз
 СТИЛЬ НАЗВ: Прості, професійні, глобальні


In [33]:
response = client.complete(
    model=model_name,
    messages=[
        {
            "role": "system", 
            "content": "You are a branding expert. Generate creative product names with brief explanations that connect to the seed words and product description."
        },
        {
            "role": "user", 
            "content": prompt
        }
    ],
        temperature=0.8,
        max_tokens=200
    )
    
print(response.choices[0].message.content)

### Супутниковий інтернет:  
1. **OrbitaLink** – Назва поєднує слова "орбіта" та "зв'язок", підкреслюючи супутникову технологію та швидкий глобальний доступ до інтернету.  
2. **SkyPulse** – Символізує "пульсацію" швидкого сигналу через супутники, що забезпечує мінімальну затримку.  
3. **GlobalSwift** – Наголос на глобальному охопленні та швидкості інтернету без меж.  
4. **SatStream** – Просте й професійне: супутниковий потік даних, що забезпечує революційну якість зв’язку.  
5. **NovaConnect** – "Нова" ера зв'язку через супутники, що об'єднує планету.

---

### Медичний монітор
